In [1]:
import polars as pl
import duckdb
import os
from pathlib import Path
import sys
sys.path.insert(1, os.path.abspath(".."))

In [2]:
from src_strategy.utils.logger import get_logger
from src_strategy.configs.dataconfig import input_output_config_3, bp_config
from src_strategy.utils.utils import extract_sys_dia_from_flowsheets
from src_strategy.data_ingest.dataloader import DataLoader

from src_strategy.outlierdetection.layer1 import Layer1PhysiologicalBound
from src_strategy.configs.outlierdetection.layer1 import physiological_bounds_config


In [3]:
dl = DataLoader(input_output_config_3, bp_config)

In [4]:
enc = dl._load_encounters()
enc = dl.cast_cols(enc)

In [6]:
enc.schema

Schema([('PrimaryMrn', Int64),
        ('PatientAgeAtAdmission', Float64),
        ('Sex', String),
        ('Ethnicity', String),
        ('FirstRace', String),
        ('MultiRacial', String),
        ('EncounterEpicCsn', Int64),
        ('AdmissionDateValue', Datetime(time_unit='us', time_zone=None)),
        ('DischargeDateValue', Datetime(time_unit='us', time_zone=None)),
        ('Arrival_Instant', Datetime(time_unit='us', time_zone=None)),
        ('FirstAdmissionOrderInstant',
         Datetime(time_unit='us', time_zone=None)),
        ('InpatientAdmissionInstant',
         Datetime(time_unit='us', time_zone=None)),
        ('Admitted_from_ED', String),
        ('PatientClass', String),
        ('InpatientAdmissionPatientClass', String),
        ('HospitalService', String),
        ('LengthOfStayInDays', Int64),
        ('AdmittingDepartment', String),
        ('DischargeDepartment', String),
        ('AdmissionType', String),
        ('AdmissionSource', String),
        ('Admi

In [27]:
enc.filter(pl.col('PatientAgeAtAdmission')<18).describe()

statistic,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN
str,f64,f64,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str
"""count""",5.0,5.0,"""5""","""5""","""5""","""5""",5.0,"""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""","""5""",5.0,"""5""","""5""",3.0,0.0,0.0,2.0,2.0,1.0,0.0,2.0,"""4""","""4""","""5""","""5""","""5""","""5""","""5""","""5"""
"""null_count""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""",2.0,5.0,5.0,3.0,3.0,4.0,5.0,3.0,"""1""","""1""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",8.3189e7,13.89295,null,null,null,null,7.0104e8,"""2024-03-21 04:48:00""","""2024-04-01 00:00:00""","""2024-03-21 19:46:48""","""2024-03-22 00:38:24""","""2024-03-21 23:42:12""",null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,114.666667,null,null,0.601333,331.285714,0.366667,null,12.342857,null,null,null,null,null,null,null,null
"""std""",1.2719e7,7.772818,null,null,null,null,1.8791e7,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,8.621678,null,null,0.139536,83.842661,null,null,1.19198,null,null,null,null,null,null,null,null
"""min""",7.3736378e7,0.0,"""Female""","""Declined""","""Black or African American""","""0""",6.79690433e8,"""2023-04-01 00:00:00""","""2023-04-03 00:00:00""","""2023-04-01 20:42:00""","""2023-04-02 02:18:00""","""2023-04-02 02:18:00""","""No""","""Inpatient""","""Emergency""","""Hospital Medicine""","""2""","""UH 05 NICU""","""UH 05 NICU""","""Emergency""","""Home & Outside Location""","""Direct Admission""","""Community acquired pneumonia""","""Sepsis following incomplete sp…","""*Unspecified""",0.0,"""NPOA-1""","""NPOA-1""",107.0,null,null,0.502667,272.0,0.366667,null,11.5,"""2023-04-01 20:43:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0"""
"""25%""",7.3831215e7,16.950034,null,null,null,null,6.86131932e8,"""2023-07-28 00:00:00""","""2023-08-02 00:00:00""","""2023-07-28 13:15:00""","""2023-07-28 17:38:00""","""2023-07-28 20:29:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,113.0,null,null,0.502667,272.0,0.366667,null,11.5,null,null,null,null,null,null,null,null
"""50%""",7.4190262e7,17.333333,null,null,null,null,6.99390548e8,"""2024-03-02 00:00:00""","""2024-04-07 00:00:00""","""2024-03-02 20:54:00""","""2024-03-02 21:28:00""","""2024-03-02 20:54:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,113.0,null,null,0.7,390.571429,0.366667,null,13.185714,null,null,null,null,null,null,null,null
"""75%""",9.5970383e7,17.341546,null,null,null,null,7.17479919e8,"""2024-12-11 00:00:00""","""2024-12-13 00:00:00""","""2024-12-11 07:03:00""","""2024-12-11 14:38:00""","""2024-12-11 07:07:00""",null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,nul

In [8]:
lab_res = dl._load_labs()
lab_res = dl.cast_cols(lab_res)

In [18]:
from collections import defaultdict
d = defaultdict(list)
for ev_grb in lab_res['Event_Grouper'].unique():
	d[ev_grb].append(lab_res.filter(pl.col('Event_Grouper') == ev_grb)['Event_Name'].unique().to_list())
print(d)

defaultdict(<class 'list'>, {'Bilirubin': [['TOTAL BILIRUBIN', 'BILIRUBIN, TOTAL']], 'Platelets': [['PLATELETS']], 'Blood Urea Nitrogen': [['BUN', 'BUN POC']], 'Sepsis Body Fluid Culture Orders - Confirmed Infection': [['AFB SMEAR', 'CULTURE', 'GRAM ST']], 'INR': [['INR']], 'Sepsis Body Fluid Culture Orders': [['UROBILINOGEN, UA', 'SPEC GRAV, UA', 'GRAM ST', 'POC URINE PH', 'CULTURE']], 'PAO2': [['PO2 ART']], 'Lactate': [['LACTIC ACID (ISTAT)', 'LACTATE']], 'Blood Culture - Confirmed Infection': [['CULTURE']], 'Bacteria in Urine - Confirmed Infection': [['BACTERIA UA']], 'WBC in Urine': [['WBC, UA']], 'WBC': [['WBC']], 'Bacteria in Urine': [['BACTERIA UA']], 'FIO2': [['FIO2']], 'Creatinine': [['CREATININE POC', 'CREATININE']], 'WBC in Urine - Confirmed Infection': [['WBC, UA']], 'Blood Culture': [['CULTURE']], 'eGFR': [['EGFR CKD EPI CR', 'EGFR CKD EPI CR POC', 'EGFR CR BEDSIDE SCHWARTZ 2009', 'EGFR']]})


In [16]:
for evn in ['WBC, UA']:
	print(lab_res.filter(pl.col("Event_Name")==evn))

shape: (21_428, 8)
┌──────────────┬─────────────┬─────────────┬─────────────┬────────────┬─────────────┬───────┬──────┐
│ EncounterEpi ┆ Event_DateT ┆ Type        ┆ Event_Group ┆ Event_Name ┆ NumericValu ┆ Value ┆ Flag │
│ cCsn         ┆ ime         ┆ ---         ┆ er          ┆ ---        ┆ e           ┆ ---   ┆ ---  │
│ ---          ┆ ---         ┆ str         ┆ ---         ┆ str        ┆ ---         ┆ str   ┆ str  │
│ i64          ┆ datetime[μs ┆             ┆ str         ┆            ┆ f64         ┆       ┆      │
│              ┆ ]           ┆             ┆             ┆            ┆             ┆       ┆      │
╞══════════════╪═════════════╪═════════════╪═════════════╪════════════╪═════════════╪═══════╪══════╡
│ 663044766    ┆ 2022-06-03  ┆ Lab Results ┆ WBC in      ┆ WBC, UA    ┆ 44.0        ┆ 44    ┆ 1    │
│              ┆ 10:41:00    ┆             ┆ Urine -     ┆            ┆             ┆       ┆      │
│              ┆             ┆             ┆ Confirmed   ┆            ┆ 

In [15]:
for evn in ['WBC']:
	print(lab_res.filter(pl.col("Event_Name")==evn))

shape: (256_418, 8)
┌──────────────┬─────────────┬─────────────┬─────────────┬────────────┬─────────────┬───────┬──────┐
│ EncounterEpi ┆ Event_DateT ┆ Type        ┆ Event_Group ┆ Event_Name ┆ NumericValu ┆ Value ┆ Flag │
│ cCsn         ┆ ime         ┆ ---         ┆ er          ┆ ---        ┆ e           ┆ ---   ┆ ---  │
│ ---          ┆ ---         ┆ str         ┆ ---         ┆ str        ┆ ---         ┆ str   ┆ str  │
│ i64          ┆ datetime[μs ┆             ┆ str         ┆            ┆ f64         ┆       ┆      │
│              ┆ ]           ┆             ┆             ┆            ┆             ┆       ┆      │
╞══════════════╪═════════════╪═════════════╪═════════════╪════════════╪═════════════╪═══════╪══════╡
│ 663001713    ┆ 2022-06-02  ┆ Lab Results ┆ WBC         ┆ WBC        ┆ 14.71       ┆ 14.71 ┆ 1    │
│              ┆ 11:53:00    ┆             ┆             ┆            ┆             ┆       ┆      │
│ 663197471    ┆ 2022-06-07  ┆ Lab Results ┆ WBC         ┆ WBC        ┆

In [12]:
for evn in ["EGFR CR BEDSIDE SCHWARTZ 2009",
	"EGFR CKD EPI CR",
	"EGFR CKD EPI CR POC",
	"EGFR"]:
	ev_vals = lab_res.filter(
		(pl.col("Event_Grouper") == "eGFR")&
		(pl.col("Event_Name") == evn)
	)['NumericValue']
	print(ev_vals)

shape: (13,)
Series: 'NumericValue' [f64]
[
	89.0
	89.0
	88.0
	93.0
	86.0
	…
	69.0
	99.0
	94.0
	86.0
	97.0
]
shape: (302_131,)
Series: 'NumericValue' [f64]
[
	7.0
	101.0
	6.0
	58.0
	52.0
	…
	98.0
	18.0
	58.0
	28.0
	83.0
]
shape: (1_739,)
Series: 'NumericValue' [f64]
[
	119.0
	85.0
	99.0
	131.0
	44.0
	…
	45.0
	124.0
	22.0
	41.0
	21.0
]
shape: (3,)
Series: 'NumericValue' [f64]
[
	103.0
	36.0
	35.0
]


In [ ]:
lab_res.filter(pl.col("Event_Grouper") == "")

In [29]:
for ev_name in lab_res['Event_Name'].unique():
	print(ev_name)
	print(lab_res.filter(pl.col("Event_Name") == ev_name)['NumericValue'].describe())

BUN POC
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 1644.0    │
│ null_count ┆ 0.0       │
│ mean       ┆ 34.011679 │
│ std        ┆ 28.132377 │
│ min        ┆ 2.7       │
│ 25%        ┆ 13.0      │
│ 50%        ┆ 25.0      │
│ 75%        ┆ 45.0      │
│ max        ┆ 126.5     │
└────────────┴───────────┘
CULTURE
shape: (2, 2)
┌────────────┬─────────┐
│ statistic  ┆ value   │
│ ---        ┆ ---     │
│ str        ┆ f64     │
╞════════════╪═════════╡
│ count      ┆ 0.0     │
│ null_count ┆ 64222.0 │
└────────────┴─────────┘
POC URINE PH
shape: (8, 2)
┌────────────┬───────┐
│ statistic  ┆ value │
│ ---        ┆ ---   │
│ str        ┆ f64   │
╞════════════╪═══════╡
│ count      ┆ 1.0   │
│ null_count ┆ 0.0   │
│ mean       ┆ 6.5   │
│ min        ┆ 6.5   │
│ 25%        ┆ 6.5   │
│ 50%        ┆ 6.5   │
│ 75%        ┆ 6.5   │
│ max        ┆ 6.5   │
└────────────┴───────┘
WB